# Lekcija 15: Fourierova transformacija i filtriranje u frekvencijskoj domeni

Svi dosadašnji filtri radili su izravno na susjedstvima piksela. Fourierova transformacija nudi potpuno drukčiji pogled: svaka se slika može zapisati kao zbroj sinusoidalnih prugastih uzoraka različitih frekvencija i orijentacija. Filtriranje se tada može provesti izravnim pojačavanjem ili potiskivanjem određenih frekvencija &mdash; a zahvaljujući **teoremu o konvoluciji**, to je matematički ista operacija kao prostorna konvolucija, samo promatrana iz druge perspektive.


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Jednodimenzionalni uvod: signal kao zbroj sinusoida

Svaki signal može se aproksimirati zbrajanjem sinusnih i kosinusnih valova različitih frekvencija. Primjerice, pravokutni val zbroj je sinusnih valova sve viših frekvencija, odnosno neparnih harmonika s opadajućom amplitudom. Fourierova transformacija djeluje u suprotnom smjeru: za zadani signal govori koje su frekvencije prisutne i koliki je doprinos svake.


In [ ]:
t = np.linspace(0, 1, 500, endpoint=False)
square_wave = np.sign(np.sin(2 * np.pi * 5 * t))

approx = np.zeros_like(t)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for n_harmonics in [1, 9]:
    approx = sum((4 / (np.pi * k)) * np.sin(2 * np.pi * 5 * k * t) for k in range(1, n_harmonics + 1, 2))
    axes[0 if n_harmonics == 1 else 1].plot(t, square_wave, '--', color='gray', label='true square wave')
    axes[0 if n_harmonics == 1 else 1].plot(t, approx, label=f'{n_harmonics} harmonic(s)')
    axes[0 if n_harmonics == 1 else 1].legend(fontsize=8)
    axes[0 if n_harmonics == 1 else 1].set_title(f'{n_harmonics} harmonic(s)', fontsize=9)
plt.tight_layout()
plt.show()

## Dvodimenzionalna Fourierova transformacija slike

Dvodimenzionalna diskretna Fourierova transformacija (DFT) rastavlja sliku na dvodimenzionalne sinusoidalne prugaste uzorke za sve kombinacije vodoravne i okomite frekvencije. Za izračun DFT-a koristi se algoritam brze Fourierove transformacije (FFT), odnosno `np.fft.fft2`, koji daje dvodimenzionalno polje kompleksnih vrijednosti s realnim i imaginarnim dijelom. Slično gradijentu slike, obično promatramo **iznos** Fourierove transformacije, koji pokazuje doprinos svake frekvencije, i **orijentaciju**. Dva uobičajena postupka prikaza jesu prikaz iznosa na logaritamskoj skali te pomicanje člana nulte frekvencije, odnosno prosječne svjetline, iz gornjeg lijevog kuta u središte.


In [ ]:
def show_spectrum(image, ax, title):
    F = np.fft.fftshift(np.fft.fft2(image)) # compute discrete Fourier transform (DFT), and shift DC value to center
    magnitude = np.log1p(np.abs(F)) # display logarithm of magnitude
    ax.imshow(magnitude, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
    return F

img = np.zeros((200, 200), dtype=np.float64)
cv2.rectangle(img, (60, 60), (140, 140), 200, -1)

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(img, cmap='gray')
axes[0].set_title('Image')
axes[0].axis('off')
show_spectrum(img, axes[1], 'Log-magnitude spectrum')
plt.tight_layout()
plt.show()

Svijetli križ kroz središte nastaje zbog oštrih vodoravnih i okomitih rubova kvadrata &mdash; nagli skokoviti rub sadrži energiju na *svakoj* frekvenciji duž smjera okomitog na rub, zbog čega oblik oštrih rubova ima tako rasprostranjen spektar.


### Prugasti uzorak izravno otkriva frekvenciju kao vrh spektra

Čisti sinusoidalni prugasti uzorak najjednostavniji je mogući test: njegov spektar trebao bi sadržavati gotovo samo jedan par svijetlih točaka, udaljenih od središta za vrijednost frekvencije, u smjeru okomitom na pruge.

Zašto *par*, a ne jedna točka? Sinusni val realnih vrijednosti, prema Eulerovoj formuli, kombinacija je dviju kompleksnih eksponencijalnih funkcija u jednakim iznosima: $\sin(2\pi f x) = \frac{1}{2i}\left(e^{i2\pi fx} - e^{-i2\pi fx}\right)$. Fourierova transformacija prve eksponencijalne funkcije daje vrh na frekvenciji $f$, a druge na $-f$. Stoga realna sinusoida stvara dva vrha na $+f$ i $-f$, simetrična oko središta nulte frekvencije (DC). Zapravo je za *svaku* sliku realnih vrijednosti iznos Fourierove transformacije simetričan: $|F(-f)|=|F(f)|$.


In [ ]:
size = 100
x = np.arange(size)
cycles_across_image = 12
grating = np.tile(np.sin(2 * np.pi * cycles_across_image * x / size), (size, 1))

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(grating, cmap='gray')
axes[0].set_title(f'Grating, {cycles_across_image} cycles across')
axes[0].axis('off')
F_grating = show_spectrum(grating, axes[1], 'Spectrum')
plt.tight_layout()
plt.show()

peak = np.unravel_index(np.argmax(np.abs(F_grating)), F_grating.shape)
print(f'brightest spectrum point at offset {peak[1] - size // 2} from center '
      f'(grating frequency was {cycles_across_image})')

## Filtriranje izmjenom spektra

**Idealni niskopropusni filtar** zadržava samo frekvencije unutar kružnog područja oko središta i zamućuje sliku; **idealni visokopropusni filtar** zadržava samo frekvencije izvan tog područja i naglašava rubove.


In [ ]:
def circular_mask(shape, radius, keep_inside=True):
    h, w = shape
    yy, xx = np.mgrid[:h, :w]
    dist = np.sqrt((yy - h / 2)**2 + (xx - w / 2)**2)
    inside = dist <= radius
    return inside if keep_inside else ~inside

photo = np.zeros((200, 200), dtype=np.float64)
cv2.rectangle(photo, (40, 40), (120, 120), 200, -1)
cv2.circle(photo, (150, 150), 30, 120, -1)

low_pass_mask = circular_mask(photo.shape, radius=15, keep_inside=True)
high_pass_mask = circular_mask(photo.shape, radius=15, keep_inside=False)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, im, title in zip(axes, [low_pass_mask, high_pass_mask],
                          ['Ideal low-pass mask\n(frequency domain)', 'Ideal high-pass mask\n(frequency domain)']):
    ax.imshow(im, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

Nakon transformacije slike u frekvencijsku domenu možemo je filtrirati izravnim maskiranjem frekvencija, a zatim vratiti inverznom transformacijom. Budući da je maska iste veličine kao slika, u frekvencijskoj domeni dovoljno je množenje element po element.


In [ ]:
def apply_frequency_filter(image, mask):
    F = np.fft.fftshift(np.fft.fft2(image)) # forward transform
    filtered = np.fft.ifft2(np.fft.ifftshift(F * mask)) # filter, then inverse transform
    return np.real(filtered)

low_pass = apply_frequency_filter(photo, low_pass_mask)
high_pass = apply_frequency_filter(photo, high_pass_mask)

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
for ax, im, title in zip(axes, [photo, low_pass, high_pass],
                          ['Original', 'Ideal low-pass\n(blurred)', 'Ideal high-pass\n(edges only)']):
    ax.imshow(im, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Cijena oštrog frekvencijskog reza: oscilacije uz rubove

Pažljivo pogledajte rezultate: od rubova oblika šire se slabe valovite oscilacije. *Idealni* filtar ima savršeno oštar rez u frekvencijskoj domeni, što prema istom teoremu o konvoluciji odgovara konvoluciji s prostornom jezgrom koja ima beskonačne vlastite oscilacije, funkcijom sinc &mdash; riječ je o **Gibbsovu fenomenu**. Gaussova maska, koja postupno opada umjesto naglog odsijecanja, izbjegava taj učinak.


In [ ]:
def gaussian_mask(shape, sigma):
    h, w = shape
    yy, xx = np.mgrid[:h, :w]
    dist2 = (yy - h / 2)**2 + (xx - w / 2)**2
    return np.exp(-dist2 / (2 * sigma**2))

smooth_low_pass = apply_frequency_filter(photo, gaussian_mask(photo.shape, sigma=15))

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(low_pass, cmap='gray')
axes[0].set_title('Ideal (sharp cutoff)\nvisible ringing near edges')
axes[1].imshow(smooth_low_pass, cmap='gray')
axes[1].set_title('Gaussian mask\nsmooth falloff, no ringing')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Teorem o konvoluciji

Zašto filtriranje u frekvencijskoj domeni radi? **Teorem o konvoluciji**

$$\mathcal{F}\{f * g\} = \mathcal{F}\{f\} \cdot \mathcal{F}\{g\}$$

kaže da je konvolucija dvaju signala u prostornoj domeni *potpuno* jednaka množenju njihovih Fourierovih transformacija element po element, nakon čega slijedi inverzna transformacija &mdash; konvolucija postaje množenje i obrnuto.


### Važan detalj: DFT daje kružnu konvoluciju

DFT implicitno promatra sliku kao da se beskonačno ponavlja, poput pločica koje se bez spojeva protežu u oba smjera. Nema „izvan slike”, nego samo „sljedeća kopija iste slike”. Zato množenje dvaju DFT-ova i inverzna transformacija ne daju običnu, „linearnu” konvoluciju iz Lekcije 10, u kojoj jezgra na kraju prelazi preko ruba u eksplicitno zadano proširenje nulama, ponavljanjem ili zrcaljenjem. Umjesto toga dobivamo **kružnu konvoluciju**, u kojoj jezgra koja prijeđe desni rub nastavlja s pikselima lijevog ruba. Rezultati se podudaraju samo ako je i prostorna obrada rubova *kružna* (`cv2.BORDER_WRAP`).


In [ ]:
rng = np.random.default_rng(0)
test_img = rng.integers(0, 255, (64, 64)).astype(np.float64)

sigma, ksize = 3, 15
ax = np.arange(ksize) - ksize // 2
gx, gy = np.meshgrid(ax, ax)
kernel = np.exp(-(gx**2 + gy**2) / (2 * sigma**2))
kernel /= kernel.sum()

# Spatial convolution, wrap-around border to match the FFT's implicit circular convolution
spatial_result = cv2.filter2D(test_img, cv2.CV_64F, kernel, borderType=cv2.BORDER_WRAP)

# Frequency-domain: pad the kernel to image size, center it at (0,0) via np.roll, then multiply spectra
kernel_padded = np.zeros_like(test_img)
kernel_padded[:ksize, :ksize] = kernel
kernel_padded = np.roll(kernel_padded, -(ksize // 2), axis=0)
kernel_padded = np.roll(kernel_padded, -(ksize // 2), axis=1)

freq_result = np.real(np.fft.ifft2(np.fft.fft2(test_img) * np.fft.fft2(kernel_padded)))

print(f'max abs difference between spatial and frequency-domain convolution: '
      f'{np.abs(spatial_result - freq_result).max():.2e}')

U praksi se filtriranje u frekvencijskoj domeni rijetko koristi u svakodnevnim zadacima računalnog vida. Većina jezgri, poput filtra srednje vrijednosti, Gaussove, Sobelove i jezgre za izoštravanje, mala je, veličine 3 × 3 ili 5 × 5, pa je izravna prostorna konvolucija već brza. Fiksni trošak dvaju izravnih FFT-ova i jednog inverznog FFT-a zato se obično ne isplati. DFT uglavnom postaje koristan kada su jezgre velike.


### Zadaci

1. Rotirajte prugasti uzorak u pokusu s vrhom spektra, tako da pruge budu dijagonalne umjesto okomite, izradom uzorka kao funkcije izraza `x*cos(theta) + y*sin(theta)`. Predvidite, a zatim provjerite kamo se pomiče vrh spektra.
2. Izradite frekvencijsku masku **pojasnopropusnog filtra** u obliku prstena: zadržite frekvencije između unutarnjeg i vanjskog polumjera, a sve ostale blokirajte. Primijenite je na `photo`. Koja vrsta sadržaja slike ostaje?
3. Isprobajte `apply_frequency_filter` s *vrlo* malom sigmom Gaussove maske, primjerice 3, i velikom, primjerice 60, na testnoj slici nalik fotografiji te opišite promjene kako sigma raste prema veličini slike.
4. Dobijte *linearnu*, a ne kružnu konvoluciju pomoću FFT-a: prije transformacije proširite i `test_img` i `kernel` nulama do najmanje `(image_size + kernel_size - 1)` duž svake osi, pomnožite spektre, primijenite inverznu transformaciju i zatim izrežite rezultat na početnu veličinu. Usporedite ga s `cv2.filter2D(..., borderType=cv2.BORDER_CONSTANT)`, koji koristi proširenje nulama, umjesto s `BORDER_WRAP`.
5. Pomoću `%timeit` usporedite izravnu konvoluciju (`cv2.filter2D`) s konvolucijom temeljenom na FFT-u, kao u prethodnom `freq_result`, za Gaussove jezgre veličine 5, 15, 31, 63 i 127 na slici veličine 256 × 256. Pri približno kojoj veličini jezgre FFT pristup postaje brži?
